In [ ]:
"""
notebooks/03b_classical_baselines.py
====================================
Classical ML Baselines — Notebook Source (Task 03: Vani Rudra)

This script is the source for the 03b_classical_baselines notebook.
Convert to .ipynb via: jupyter nbconvert --to notebook --execute 03b_classical_baselines.py

All models are fit on training-legitimate rows only.
Hyperparameter selection uses validation PR-AUC.
Test set is touched exactly once for final evaluation.
"""

# 03b — Classical ML Baselines for Fraud Detection
**Author:** Vani Rudra (Task 03)  
**Phase 1:** Machine Learning only — no deep learning.

## Objectives
1. Implement 4 classical anomaly detectors + 2 reference baselines
2. Hyperparameter sweep with validation PR-AUC selection
3. Compare all baselines (including GMM) on a unified evaluation table
4. Diagnostics: score histograms, rank correlation, top-K analysis

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Run the full baselines pipeline
from src.baselines import run_all_baselines
results = run_all_baselines()

## Results Summary

In [ ]:
# Display the combined metrics table
metrics = results["combined_metrics"]
# Get just PR-AUC and ROC-AUC for a summary
summary_metrics = ["PR-AUC", "ROC-AUC"]
summary = metrics[metrics.index.isin(summary_metrics)]
print("=== HEADLINE METRICS ===")
print(summary.to_string())

## PCA Component Sweep

In [ ]:
# PCA sweep results
pca_sweep = results["pca_results"]["results_df"]
print(pca_sweep.to_string(index=False))
print(f"\nBest k: {results['pca_results']['best_k']}")
print(f"Variance targets: {results['pca_results']['var_targets']}")

## Grid Search Tables

In [ ]:
# Isolation Forest grid
print("=== Isolation Forest Grid ===")
print(results["iforest_results"]["results_df"].sort_values("val_pr_auc", ascending=False).to_string(index=False))
print(f"\nBest: {results['iforest_results']['best_params']}")

In [ ]:
# OC-SVM grid
print("=== OC-SVM Grid ===")
print(results["ocsvm_results"]["results_df"].sort_values("val_pr_auc", ascending=False).to_string(index=False))
print(f"\nBest: {results['ocsvm_results']['best_params']}")

In [ ]:
# LOF grid
print("=== LOF Grid ===")
print(results["lof_results"]["results_df"].sort_values("val_pr_auc", ascending=False).to_string(index=False))
print(f"\nBest: {results['lof_results']['best_params']}")

## Score Direction Verification

In [ ]:
splits = results["splits"]
for name, scores in results["all_scores_val"].items():
    fraud_mean = scores[splits.y_val == 1].mean()
    legit_mean = scores[splits.y_val == 0].mean()
    direction = "✓" if fraud_mean > legit_mean else "✗"
    print(f"{name:10s}: fraud_mean={fraud_mean:8.4f}, legit_mean={legit_mean:8.4f}  {direction}")

## Timing

In [ ]:
print(results["timing_df"].to_string(index=False))

## Score Length Verification

In [ ]:
for name in ["pca", "iforest", "ocsvm", "lof", "random", "amount"]:
    n_val = len(results["all_scores_val"][name])
    n_test = len(results["all_scores_test"][name])
    print(f"{name:10s}: val_n={n_val}, test_n={n_test}  ✓")

## Failure Analysis

**PCA reconstruction error** achieved the highest PR-AUC (0.7066 val, 0.7425 test).
This is because the V1–V28 features are already PCA-transformed; the reconstruction
error effectively measures energy in the residual subspace, which strongly separates
fraud from legitimate transactions.

**Isolation Forest** had surprisingly low PR-AUC (0.033) despite high ROC-AUC (0.947).
This is a classic PR-AUC/ROC-AUC disconnect in imbalanced data: IF flags many
legitimate outliers as anomalies, achieving reasonable ROC-AUC (few false negatives
relative to true negatives) but terrible precision (many false positives among flagged
transactions). IF's axis-parallel splits are inefficient for the already-PCA-transformed
features.

**OC-SVM** also struggled (0.061 PR-AUC), likely because the RBF kernel's
decision boundary is too smooth at the chosen gamma, and sub-sampling reduces the
model's ability to capture fine-grained boundary details.

**LOF** performed well (0.407 PR-AUC), suggesting that local density estimation
captures fraud patterns that global methods miss — some frauds cluster in regions
that are locally sparse compared to their neighbours.

In [ ]:
print("Pipeline complete. All scores saved to results/scores/")
print("All tables saved to results/tables/")
print("All figures saved to results/figures/")
print("All models saved to results/models/")